# Brain Tumor Classification, Lightweight CNN (Colab reproduction)

This notebook reconstructs the end-to-end pipeline described in the draft paper (preprocessing → lightweight CNN → training callbacks → evaluation).

**Targets (from the draft):** 224×224 inputs, Adam (lr=1e-3), 60 epochs, batch size 32, sparse categorical cross-entropy, ReduceLROnPlateau (factor=0.5, patience=5), ModelCheckpoint on validation accuracy, early stopping and a custom stop at 100% train+val accuracy.

Datasets referenced by the project:
- Zenodo: DOI 10.5281/zenodo.17447727
- Mendeley: DOI 10.17632/zwr4ntf94j.5

Note: Download access policies may differ across providers. Zenodo is used as the default download path because it is straightforward from Colab.


In [9]:
# ============================
# Cell: One-time FAST extraction for Zenodo RAR
# Strategy: extract to /content (local SSD, fast), then copy once to Google Drive cache
# ============================
import shutil
import subprocess
import time
from pathlib import Path

# --- Paths (Drive + local) ---
CACHE_DIR = Path("/content/drive/MyDrive/BT_paper_ready/data_cache")
RAR_PATH = CACHE_DIR / "Brain_Tumor_Classiifaction_zenodo17447727.rar"

# Final persistent cache on Drive (used by all future runs)
DRIVE_DATA_ROOT = CACHE_DIR / "BT_ZENODO_17447727"

# Temporary local extraction folder (fast I/O)
LOCAL_EXTRACT_ROOT = Path("/content/_bt_extract_tmp")

def run(cmd):
    print("\n$", " ".join(cmd))
    subprocess.run(cmd, check=True)

# --- Sanity checks ---
print("RAR_PATH:", RAR_PATH)
if not RAR_PATH.exists():
    raise FileNotFoundError(f"RAR not found at: {RAR_PATH}")

print("RAR size (MB):", round(RAR_PATH.stat().st_size / (1024**2), 2))
print("LOCAL_EXTRACT_ROOT:", LOCAL_EXTRACT_ROOT)
print("DRIVE_DATA_ROOT:", DRIVE_DATA_ROOT)

# --- Install extraction tools (7z + rar support) ---
run(["bash", "-lc", "apt-get update -y >/dev/null 2>&1"])
run(["bash", "-lc", "apt-get install -y p7zip-full p7zip-rar >/dev/null 2>&1 || apt-get install -y p7zip-full >/dev/null 2>&1"])

# --- Clean targets (so we never mix partial extractions) ---
if LOCAL_EXTRACT_ROOT.exists():
    shutil.rmtree(LOCAL_EXTRACT_ROOT, ignore_errors=True)
LOCAL_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)

if DRIVE_DATA_ROOT.exists():
    shutil.rmtree(DRIVE_DATA_ROOT, ignore_errors=True)

# --- Optional: list archive content quickly to confirm readability ---
print("\nListing archive (first ~120 lines) to confirm it is readable...")
run(["bash", "-lc", f"7z l '{RAR_PATH}' | head -n 120"])

# --- Extract locally (FAST) ---
print("\nExtracting to local /content (fast I/O)...")
t0 = time.time()
run(["bash", "-lc", f"7z x -y -o'{LOCAL_EXTRACT_ROOT}' '{RAR_PATH}'"])
print(f"Local extraction completed in {time.time() - t0:.1f} seconds")

# --- Quick progress check: count extracted files/images ---
print("\nCounting extracted files (this may take a few seconds)...")
run(["bash", "-lc", f"find '{LOCAL_EXTRACT_ROOT}' -type f | wc -l"])
run(["bash", "-lc", f"find '{LOCAL_EXTRACT_ROOT}' -type f \\( -iname '*.png' -o -iname '*.jpg' -o -iname '*.jpeg' -o -iname '*.bmp' -o -iname '*.tif' -o -iname '*.tiff' \\) | wc -l"])

# --- Copy to Drive cache (ONE-TIME) ---
print("\nCopying extracted dataset to Google Drive cache (one-time)...")
t1 = time.time()
shutil.copytree(LOCAL_EXTRACT_ROOT, DRIVE_DATA_ROOT)
print(f"Drive copy completed in {time.time() - t1:.1f} seconds")

print("\nDrive cache is ready at:", DRIVE_DATA_ROOT)
print("From the next run onward, you should NOT re-download or re-extract. Just reuse this cache.")


RAR_PATH: /content/drive/MyDrive/BT_paper_ready/data_cache/Brain_Tumor_Classiifaction_zenodo17447727.rar
RAR size (MB): 164.37
LOCAL_EXTRACT_ROOT: /content/_bt_extract_tmp
DRIVE_DATA_ROOT: /content/drive/MyDrive/BT_paper_ready/data_cache/BT_ZENODO_17447727

$ bash -lc apt-get update -y >/dev/null 2>&1

$ bash -lc apt-get install -y p7zip-full p7zip-rar >/dev/null 2>&1 || apt-get install -y p7zip-full >/dev/null 2>&1

Listing archive (first ~120 lines) to confirm it is readable...

$ bash -lc 7z l '/content/drive/MyDrive/BT_paper_ready/data_cache/Brain_Tumor_Classiifaction_zenodo17447727.rar' | head -n 120

Extracting to local /content (fast I/O)...

$ bash -lc 7z x -y -o'/content/_bt_extract_tmp' '/content/drive/MyDrive/BT_paper_ready/data_cache/Brain_Tumor_Classiifaction_zenodo17447727.rar'
Local extraction completed in 22.1 seconds

Counting extracted files (this may take a few seconds)...

$ bash -lc find '/content/_bt_extract_tmp' -type f | wc -l

$ bash -lc find '/content/_bt_extr

In [10]:
# ============================
# Cell: Paper-ready training pipeline (single cell) assuming Drive cache is ALREADY built
# Uses cached extracted dataset on Drive, copies to /content for fast A100 training, saves outputs back to Drive
# ============================
import os, json, time, shutil, random
import numpy as np
import tensorflow as tf
from pathlib import Path

# ----------------------------
# 0) Drive paths (cache already exists)
# ----------------------------
PROJECT_DIR = Path("/content/drive/MyDrive/BT_paper_ready")
CACHE_DIR = PROJECT_DIR / "data_cache"
RUNS_DIR = PROJECT_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# The cache root created by your extraction cell
DRIVE_DATA_ROOT = CACHE_DIR / "BT_ZENODO_17447727"

# ----------------------------
# 1) Reproducibility + A100 speed knobs
# ----------------------------
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    tf.config.experimental.enable_tensor_float_32_execution(True)
    print("TF32 enabled.")
except Exception as e:
    print("TF32 not available:", e)

from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")
print("Mixed precision policy:", mixed_precision.global_policy())

AUTOTUNE = tf.data.AUTOTUNE

# ----------------------------
# 2) Paper-targeted settings (Zenodo notes 224x224)
# ----------------------------
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS_MAX = 60
BASE_LR = 1e-3
VAL_BATCH_SIZE = 256  # faster validation batching (same samples)

# ----------------------------
# 3) Locate split root inside the extracted cache
# Zenodo description suggests: BrainMRI_Classification/{train,validation,test}
# But we also search robustly in case extraction created a different top folder.
# ----------------------------
IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def has_images(p: Path) -> bool:
    if not p.exists() or not p.is_dir():
        return False
    for f in p.rglob("*"):
        if f.is_file() and f.suffix.lower() in IMG_EXTS:
            return True
    return False

def is_class_root(p: Path) -> bool:
    if not p.exists() or not p.is_dir():
        return False
    subs = [d for d in p.iterdir() if d.is_dir()]
    if len(subs) < 2:
        return False
    good = 0
    for d in subs:
        if has_images(d):
            good += 1
        if good >= 2:
            return True
    return False

def find_split_root(root: Path):
    """
    Find a directory that contains:
      - train/ (class-root)
      - validation/ OR valid/ OR val/ (class-root)
      - test/ (class-root)
    """
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for p in candidates:
        train = p / "train"
        valid = p / "validation"
        valid_alt1 = p / "valid"
        valid_alt2 = p / "val"
        test = p / "test"
        if train.exists() and test.exists() and is_class_root(train) and is_class_root(test):
            if valid.exists() and is_class_root(valid):
                return p, "validation"
            if valid_alt1.exists() and is_class_root(valid_alt1):
                return p, "valid"
            if valid_alt2.exists() and is_class_root(valid_alt2):
                return p, "val"
    return None, None

if not DRIVE_DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Drive cache folder not found: {DRIVE_DATA_ROOT}\n"
        f"Please run the extraction cell first."
    )

SPLIT_ROOT, VALID_FOLDER_NAME = find_split_root(DRIVE_DATA_ROOT)
if SPLIT_ROOT is None:
    # Print a shallow tree for diagnosis
    print("\n[DEBUG] Could not find train/validation/test under:", DRIVE_DATA_ROOT)
    tops = sorted([p for p in DRIVE_DATA_ROOT.iterdir() if p.is_dir()])
    print("Top-level folders:", [t.name for t in tops[:50]])
    raise RuntimeError("Could not locate split root with train/valid(or val)/test inside the Drive cache.")

print("SPLIT_ROOT:", SPLIT_ROOT)
print("VALID_FOLDER_NAME:", VALID_FOLDER_NAME)

# Normalize validation folder name to "validation" inside Drive cache to keep everything consistent
if VALID_FOLDER_NAME != "validation":
    src = SPLIT_ROOT / VALID_FOLDER_NAME
    dst = SPLIT_ROOT / "validation"
    if not dst.exists():
        src.rename(dst)
        print(f"Renamed {VALID_FOLDER_NAME}/ -> validation/ in SPLIT_ROOT for consistency.")
    VALID_FOLDER_NAME = "validation"

TRAIN_DIR_DRIVE = SPLIT_ROOT / "train"
VAL_DIR_DRIVE = SPLIT_ROOT / "validation"
TEST_DIR_DRIVE = SPLIT_ROOT / "test"

# ----------------------------
# 4) Copy dataset from Drive to /content (fast local I/O)
# ----------------------------
LOCAL_ROOT = Path("/content/bt_data")
LOCAL_SPLIT_ROOT = LOCAL_ROOT / "BrainMRI_Classification"
if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT, ignore_errors=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

print("Copying dataset from Drive cache to /content (fast I/O)...")
shutil.copytree(SPLIT_ROOT, LOCAL_SPLIT_ROOT)

TRAIN_DIR = LOCAL_SPLIT_ROOT / "train"
VAL_DIR   = LOCAL_SPLIT_ROOT / "validation"
TEST_DIR  = LOCAL_SPLIT_ROOT / "test"

print("TRAIN_DIR:", TRAIN_DIR)
print("VAL_DIR:", VAL_DIR)
print("TEST_DIR:", TEST_DIR)

# ----------------------------
# 5) Build tf.data datasets
# ----------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    str(TRAIN_DIR),
    labels="inferred",
    label_mode="categorical",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    str(VAL_DIR),
    labels="inferred",
    label_mode="categorical",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    str(TEST_DIR),
    labels="inferred",
    label_mode="categorical",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print("CLASS_NAMES:", CLASS_NAMES)
print("NUM_CLASSES:", NUM_CLASSES)
assert NUM_CLASSES >= 2, "NUM_CLASSES < 2. Something is wrong with the dataset root."

train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
test_ds  = test_ds.cache().prefetch(AUTOTUNE)

val_ds_fast  = val_ds.unbatch().batch(VAL_BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)
test_ds_fast = test_ds.unbatch().batch(VAL_BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)

# ----------------------------
# 6) Model: paper-like CNN + augmentation
# ----------------------------
from tensorflow.keras import layers, models

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.1),
], name="data_augmentation")

def build_cnn(input_shape=(224, 224, 3), num_classes=4):
    inputs = layers.Input(shape=input_shape)
    x = layers.Rescaling(1./255)(inputs)
    x = data_augmentation(x)

    for filters, drop in [(32, 0.25), (64, 0.25), (128, 0.25), (256, 0.30)]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(drop)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50)(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)

    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
    return models.Model(inputs, outputs)

model = build_cnn(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)

# ----------------------------
# 7) Compile + callbacks + train
# ----------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=BASE_LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
    steps_per_execution=64
)

# Run folder (Drive)
run_id = time.strftime("%Y%m%d_%H%M%S")
OUT_DIR = RUNS_DIR / f"run_{run_id}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_CKPT = Path("/content/best_model.keras")
DRIVE_CKPT = OUT_DIR / "best_model.keras"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(LOCAL_CKPT),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        min_delta=1e-3,
        restore_best_weights=True,
        verbose=1
    ),
]

run_config = {
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size_train": BATCH_SIZE,
    "batch_size_val": VAL_BATCH_SIZE,
    "epochs_max": EPOCHS_MAX,
    "base_lr": BASE_LR,
    "class_names": CLASS_NAMES,
    "tf_version": tf.__version__,
    "mixed_precision": str(mixed_precision.global_policy()),
    "drive_cache_root": str(SPLIT_ROOT),
}
with open(OUT_DIR / "run_config.json", "w") as f:
    json.dump(run_config, f, indent=2)

t0 = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds_fast,
    epochs=EPOCHS_MAX,
    callbacks=callbacks,
    verbose=1
)
print(f"Training time (s): {time.time() - t0:.1f}")

# Persist best model on Drive
if LOCAL_CKPT.exists():
    shutil.copy2(LOCAL_CKPT, DRIVE_CKPT)
    print("Saved best model to Drive:", DRIVE_CKPT)

with open(OUT_DIR / "history.json", "w") as f:
    json.dump(history.history, f, indent=2)

# ----------------------------
# 8) Evaluate + artifacts (paper-ready)
# ----------------------------
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt

best_model = tf.keras.models.load_model(str(DRIVE_CKPT))
test_loss, test_acc = best_model.evaluate(test_ds_fast, verbose=1)

with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump({"test_loss": float(test_loss), "test_accuracy": float(test_acc)}, f, indent=2)

y_true, y_pred = [], []
for xb, yb in test_ds_fast:
    probs = best_model.predict(xb, verbose=0)
    y_true.extend(tf.argmax(yb, axis=1).numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
cm = confusion_matrix(y_true, y_pred)

with open(OUT_DIR / "classification_report_TEST.txt", "w") as f:
    f.write(report)

cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.to_csv(OUT_DIR / "confusion_matrix_TEST.csv", index=True)

plt.figure(figsize=(7, 6))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix (TEST)")
plt.xticks(np.arange(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
plt.yticks(np.arange(NUM_CLASSES), CLASS_NAMES)
plt.colorbar()
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, int(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.savefig(OUT_DIR / "confusion_matrix_TEST.png", dpi=200)
plt.close()

print("Done. Outputs in:", OUT_DIR)


TF32 enabled.
Mixed precision policy: <DTypePolicy "mixed_float16">
SPLIT_ROOT: /content/drive/MyDrive/BT_paper_ready/data_cache/BT_ZENODO_17447727/BRISC_TUMOR_20000(Reso)_Set
VALID_FOLDER_NAME: valid
Renamed valid/ -> validation/ in SPLIT_ROOT for consistency.
Copying dataset from Drive cache to /content (fast I/O)...
TRAIN_DIR: /content/bt_data/BrainMRI_Classification/train
VAL_DIR: /content/bt_data/BrainMRI_Classification/validation
TEST_DIR: /content/bt_data/BrainMRI_Classification/test
Found 20000 files belonging to 4 classes.
Found 2142 files belonging to 4 classes.
Found 2311 files belonging to 4 classes.
CLASS_NAMES: ['Normal_no tumor', 'glioma', 'meningioma', 'pituitary']
NUM_CLASSES: 4
Epoch 1/60
577/625 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.5499 - loss: 1.1769
Epoch 1: val_accuracy improved from -inf to 0.33240, saving model to /content/best_model.keras


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


625/625 ━━━━━━━━━━━━━━━━━━━━ 59s 48ms/step - accuracy: 0.5535 - loss: 1.1656 - val_accuracy: 0.3324 - val_loss: 2.8931 - learning_rate: 0.0010
Epoch 2/60
577/625 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.6836 - loss: 0.7720
Epoch 2: val_accuracy improved from 0.33240 to 0.73203, saving model to /content/best_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 30ms/step - accuracy: 0.6846 - loss: 0.7700 - val_accuracy: 0.7320 - val_loss: 0.6991 - learning_rate: 0.0010
Epoch 3/60
577/625 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.7342 - loss: 0.6649
Epoch 3: val_accuracy did not improve from 0.73203
625/625 ━━━━━━━━━━━━━━━━━━━━ 20s 29ms/step - accuracy: 0.7361 - loss: 0.6605 - val_accuracy: 0.6657 - val_loss: 1.3924 - learning_rate: 0.0010
Epoch 4/60
577/625 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step - accuracy: 0.8173 - loss: 0.4765
Epoch 4: val_accuracy improved from 0.73203 to 0.86648, saving model to /content/best_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 21s 30ms/step - accuracy: 0.8181 -

In [11]:
# ============================
# Cell: Post-training evaluation + paper-style statistics export (single cell)
# - Loads the best model saved on Drive
# - Evaluates on TEST (or VAL if TEST not found)
# - Prints: loss/accuracy, classification report, confusion matrix
# - Saves: metrics.json, classification_report.txt, confusion_matrix.csv/png in the same run folder
# ============================
import json
import numpy as np
import tensorflow as tf
from pathlib import Path

from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# 1) Locate the latest run folder and best model
# ----------------------------
RUNS_DIR = Path("/content/drive/MyDrive/BT_paper_ready/runs")
if not RUNS_DIR.exists():
    raise FileNotFoundError(f"RUNS_DIR not found: {RUNS_DIR}")

run_folders = sorted([p for p in RUNS_DIR.iterdir() if p.is_dir() and p.name.startswith("run_")])
if not run_folders:
    raise RuntimeError(f"No run_ folders found under: {RUNS_DIR}")

LATEST_RUN = run_folders[-1]
BEST_MODEL_PATH = LATEST_RUN / "best_model.keras"

print("LATEST_RUN:", LATEST_RUN)
print("BEST_MODEL_PATH:", BEST_MODEL_PATH)

if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"best_model.keras not found in: {LATEST_RUN}")

# ----------------------------
# 2) Locate the local dataset copy used for training
# ----------------------------
LOCAL_SPLIT_ROOT = Path("/content/bt_data/BrainMRI_Classification")
TRAIN_DIR = LOCAL_SPLIT_ROOT / "train"
VAL_DIR = LOCAL_SPLIT_ROOT / "validation"
TEST_DIR = LOCAL_SPLIT_ROOT / "test"

if not LOCAL_SPLIT_ROOT.exists():
    raise FileNotFoundError(
        f"Local dataset root not found: {LOCAL_SPLIT_ROOT}\n"
        f"Run the training cell first (it copies the Drive cache to /content)."
    )

print("LOCAL_SPLIT_ROOT:", LOCAL_SPLIT_ROOT)
print("TRAIN_DIR exists:", TRAIN_DIR.exists())
print("VAL_DIR exists:", VAL_DIR.exists())
print("TEST_DIR exists:", TEST_DIR.exists())

# ----------------------------
# 3) Recreate datasets exactly as in training (paper-consistent)
# ----------------------------
# Keep these aligned with the training cell
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
VAL_BATCH_SIZE = 256
AUTOTUNE = tf.data.AUTOTUNE

val_ds = tf.keras.utils.image_dataset_from_directory(
    str(VAL_DIR),
    labels="inferred",
    label_mode="categorical",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = None
eval_name = "VAL"
eval_ds = None

if TEST_DIR.exists():
    test_ds = tf.keras.utils.image_dataset_from_directory(
        str(TEST_DIR),
        labels="inferred",
        label_mode="categorical",
        seed=SEED,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False
    )
    eval_name = "TEST"
    eval_ds = test_ds
else:
    eval_ds = val_ds

CLASS_NAMES = val_ds.class_names  # consistent ordering
NUM_CLASSES = len(CLASS_NAMES)
print("CLASS_NAMES:", CLASS_NAMES)
print("NUM_CLASSES:", NUM_CLASSES)

# Cache/prefetch and fast batching for evaluation
eval_ds = eval_ds.cache().prefetch(AUTOTUNE)
eval_ds_fast = eval_ds.unbatch().batch(VAL_BATCH_SIZE, drop_remainder=False).prefetch(AUTOTUNE)

# ----------------------------
# 4) Load best model and evaluate
# ----------------------------
best_model = tf.keras.models.load_model(str(BEST_MODEL_PATH))

loss, acc = best_model.evaluate(eval_ds_fast, verbose=1)
print(f"\nBest checkpoint evaluation on {eval_name}:")
print(f"  {eval_name}_loss: {loss:.4f}")
print(f"  {eval_name}_accuracy: {acc:.4f}")

metrics_out = {f"{eval_name.lower()}_loss": float(loss), f"{eval_name.lower()}_accuracy": float(acc)}
with open(LATEST_RUN / "metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)

# ----------------------------
# 5) Predictions -> classification report + confusion matrix
# ----------------------------
y_true, y_pred = [], []

for xb, yb in eval_ds_fast:
    probs = best_model.predict(xb, verbose=0)
    y_true.extend(tf.argmax(yb, axis=1).numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
cm = confusion_matrix(y_true, y_pred)

print(f"\nClassification Report ({eval_name}):\n{report}")
print(f"Confusion Matrix ({eval_name}):\n{cm}")

# Save report
report_path = LATEST_RUN / f"classification_report_{eval_name}.txt"
with open(report_path, "w") as f:
    f.write(report)

# Save confusion matrix CSV
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_csv_path = LATEST_RUN / f"confusion_matrix_{eval_name}.csv"
cm_df.to_csv(cm_csv_path, index=True)

# Save confusion matrix PNG
plt.figure(figsize=(7, 6))
plt.imshow(cm, interpolation="nearest")
plt.title(f"Confusion Matrix ({eval_name})")
plt.xticks(np.arange(NUM_CLASSES), CLASS_NAMES, rotation=45, ha="right")
plt.yticks(np.arange(NUM_CLASSES), CLASS_NAMES)
plt.colorbar()
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        plt.text(j, i, int(cm[i, j]), ha="center", va="center")
plt.tight_layout()
cm_png_path = LATEST_RUN / f"confusion_matrix_{eval_name}.png"
plt.savefig(cm_png_path, dpi=200)
plt.close()

print("\nSaved artifacts:")
print(" -", LATEST_RUN / "metrics.json")
print(" -", report_path)
print(" -", cm_csv_path)
print(" -", cm_png_path)
print("\nDone. Outputs in:", LATEST_RUN)


LATEST_RUN: /content/drive/MyDrive/BT_paper_ready/runs/run_20251217_004759
BEST_MODEL_PATH: /content/drive/MyDrive/BT_paper_ready/runs/run_20251217_004759/best_model.keras
LOCAL_SPLIT_ROOT: /content/bt_data/BrainMRI_Classification
TRAIN_DIR exists: True
VAL_DIR exists: True
TEST_DIR exists: True
Found 2142 files belonging to 4 classes.
Found 2311 files belonging to 4 classes.
CLASS_NAMES: ['Normal_no tumor', 'glioma', 'meningioma', 'pituitary']
NUM_CLASSES: 4
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.9900 - loss: 0.0505

Best checkpoint evaluation on TEST:
  TEST_loss: 0.0505
  TEST_accuracy: 0.9900


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Classification Report (TEST):
                 precision    recall  f1-score   support

Normal_no tumor     0.9819    0.9982    0.9900       545
         glioma     0.9945    0.9856    0.9900       554
     meningioma     0.9885    0.9804    0.9844       612
      pituitary     0.9950    0.9967    0.9958       600

       accuracy                         0.9900      2311
      macro avg     0.9900    0.9902    0.9901      2311
   weighted avg     0.9901    0.9900    0.9900      2311

Confusion Matrix (TEST):
[[544   1   0   0]
 [  0 546   5   3]
 [ 10   2 600   0]
 [  0   0   2 598]]

Saved artifacts:
 - /content/drive/MyDrive/BT_paper_ready/runs/run_20251217_004759/metrics.json
 - /content/drive/MyDrive/BT_paper_ready/runs/run_20251217_004759/classification_report_TEST.txt
 - /content/drive/MyDrive/BT_paper_ready/runs/run_20251217_004759/confusion_matrix_TEST.csv
 - /content/drive/MyDrive/BT_paper_ready/runs/run_20251217_004759/confusion_matrix_TEST.png

Done. Outputs in: /content/dr

## Reproducibility checklist
- Confirm class mapping order equals the paper's convention (Glioma, Meningioma, No Tumor, Pituitary). If not, reorder labels consistently for reporting.
- Confirm image size (224 vs 256). The draft text contains both values in different sections.
- Confirm which split was used for the headline metrics. The confusion-matrix totals in the draft align with the reported validation split size.
- Fix random seeds and keep deterministic ops enabled for tight run-to-run variance.
